# Route 53 & DNS

## DNS Basics

**DNS (Domain Name System)** translates human-readable domain names (example.com) to IP addresses (192.0.2.1). When you type a URL in your browser, DNS resolves it to an IP address.

**Route 53** is AWS's managed DNS service. It's highly available, scalable, and integrates with other AWS services.

## Hosted Zones

A **hosted zone** is a container for DNS records for a specific domain. When you register a domain with Route 53 or transfer it from another registrar, you create a hosted zone.

Route 53 provides name servers for your hosted zone. You update your domain registrar to point to these name servers.

## Record Types

**A records** map domain names to IPv4 addresses.

**AAAA records** map domain names to IPv6 addresses.

**CNAME records** create aliases to other domain names (e.g., www.example.com → example.com).

**MX records** specify mail servers for email delivery.

**TXT records** store text data (e.g., SPF, DKIM for email authentication).

**NS records** specify name servers for a domain.

**SOA records** contain zone authority information.

## Routing Policies

**Simple routing** directs traffic to a single resource.

**Weighted routing** distributes traffic based on weights (e.g., 70% to one resource, 30% to another).

**Latency-based routing** directs traffic to the resource with lowest latency.

**Failover routing** directs traffic to a primary resource; if it fails, traffic goes to a secondary resource.

**Geolocation routing** directs traffic based on geographic location.

**Multi-value answer routing** returns multiple IP addresses randomly.

## Health Checks

**Health checks** monitor the health of your resources. If a resource fails a health check, Route 53 stops routing traffic to it.

Health checks can monitor HTTP endpoints, TCP connections, or CloudWatch alarms.

## Hands-On: Create Hosted Zone and Records

Create a hosted zone:

```bash
aws route53 create-hosted-zone --name example.com \
  --caller-reference $(date +%s)
```

List hosted zones:

```bash
aws route53 list-hosted-zones
```

Create an A record:

```bash
aws route53 change-resource-record-sets --hosted-zone-id ZONE_ID \
  --change-batch '{
    "Changes": [
      {
        "Action": "CREATE",
        "ResourceRecordSet": {
          "Name": "example.com",
          "Type": "A",
          "TTL": 300,
          "ResourceRecords": [{"Value": "192.0.2.1"}]
        }
      }
    ]
  }'
```

Create a CNAME record:

```bash
aws route53 change-resource-record-sets --hosted-zone-id ZONE_ID \
  --change-batch '{
    "Changes": [
      {
        "Action": "CREATE",
        "ResourceRecordSet": {
          "Name": "www.example.com",
          "Type": "CNAME",
          "TTL": 300,
          "ResourceRecords": [{"Value": "example.com"}]
        }
      }
    ]
  }'
```

Create a health check:

```bash
aws route53 create-health-check --health-check-config '{
  "Type": "HTTP",
  "ResourcePath": "/health",
  "FullyQualifiedDomainName": "example.com",
  "Port": 80,
  "RequestInterval": 30,
  "FailureThreshold": 3
}'
```

## Python Boto3 Example

In [ ]:
import boto3

route53 = boto3.client('route53')

# Create hosted zone
response = route53.create_hosted_zone(
    Name='example.com',
    CallerReference='unique-ref-123'
)
zone_id = response['HostedZone']['Id']

# Create A record
route53.change_resource_record_sets(
    HostedZoneId=zone_id,
    ChangeBatch={
        'Changes': [
            {
                'Action': 'CREATE',
                'ResourceRecordSet': {
                    'Name': 'example.com',
                    'Type': 'A',
                    'TTL': 300,
                    'ResourceRecords': [{'Value': '192.0.2.1'}]
                }
            }
        ]
    }
)

# List records
response = route53.list_resource_record_sets(HostedZoneId=zone_id)
for record in response['ResourceRecordSets']:
    print(f"{record['Name']} ({record['Type']})")

## Terraform Example

```hcl
resource "aws_route53_zone" "main" {
  name = "example.com"
}

resource "aws_route53_record" "www" {
  zone_id = aws_route53_zone.main.zone_id
  name    = "www.example.com"
  type    = "A"
  ttl     = 300
  records = ["192.0.2.1"]
}

resource "aws_route53_record" "alias" {
  zone_id = aws_route53_zone.main.zone_id
  name    = "example.com"
  type    = "A"

  alias {
    name                   = aws_cloudfront_distribution.s3.domain_name
    zone_id                = aws_cloudfront_distribution.s3.hosted_zone_id
    evaluate_target_health = false
  }
}

resource "aws_route53_health_check" "main" {
  fqdn              = "example.com"
  port              = 80
  type              = "HTTP"
  resource_path     = "/health"
  failure_threshold = 3
  request_interval  = 30
}
```

## Alias Records

**Alias records** are Route 53-specific records that map to AWS resources (CloudFront, ELB, S3 websites). Unlike CNAME records, alias records can be created at the zone apex (example.com).

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is Route 53?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="0">
      <span>AWS's managed DNS service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="1">
      <span>A database service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="2">
      <span>A compute service</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="3">
      <span>A storage service</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is a hosted zone?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="0">
      <span>A DNS record</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="1">
      <span>A container for DNS records for a specific domain</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="2">
      <span>A health check</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5738291" value="3">
      <span>A routing policy</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What does an A record do?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="0">
      <span>Maps domain names to IPv6 addresses</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="1">
      <span>Creates an alias to another domain</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="2">
      <span>Maps domain names to IPv4 addresses</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9384756" value="3">
      <span>Specifies mail servers</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is latency-based routing?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="0">
      <span>Directs traffic to the resource with lowest latency</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="1">
      <span>Directs traffic based on geographic location</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="2">
      <span>Distributes traffic equally</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4729183" value="3">
      <span>Directs traffic to a primary resource only</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is a health check in Route 53?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="0">
      <span>A DNS record</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="1">
      <span>A monitor that checks resource health and stops routing if it fails</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="2">
      <span>A routing policy</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8293847" value="3">
      <span>A hosted zone</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>